# Module 1: QLoRA Fine-Tuning — Llama 3.1 8B

**Certification:** AWS Certified Machine Learning Engineer – Associate (MLA-C01)  
**Estimated Time:** 2–3 hours  
**Prerequisites:** ml.g5.2xlarge notebook instance, JSONL training data in S3, baseline metrics  
**Services Covered:** Amazon SageMaker, Amazon S3, Amazon CloudWatch  
**Key Libraries:** transformers, peft, bitsandbytes, trl, datasets

## Step 1: Environment Setup and Dependency Check

In [ ]:
# Cell: Pin all ML packages to a compatible version set
import subprocess
import sys

packages = [
    "transformers==4.45.2",
    "trl==0.11.4",
    "peft==0.12.0",
    "accelerate==0.34.0",
    "datasets==3.0.0",
    "bitsandbytes==0.43.3",
]

for pkg in packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", pkg]
    )

print("All ML dependencies pinned to compatible versions.")
print("Ignoring non-critical opencv-python/pathos warnings.")
print("⚠️  Go to Kernel → Restart Kernel, then re-run from the imports cell.")

In [ ]:
import torch
import json
import time
from pathlib import Path
from torch.utils.data import DataLoader
from torch.nn import CrossEntropyLoss
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model,
    PeftModel,
)
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

# ── Paths ──
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
DATA_PATH = Path("artifacts/training_data.jsonl")
OUTPUT_DIR = Path("./qlora-output")
ADAPTER_DIR = Path("./llama-3-qlora-final")
BASELINE_FILE = Path("./baseline_metrics.json")

OUTPUT_DIR.mkdir(exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model:          {MODEL_ID}")
print(f"Data path:      {DATA_PATH.resolve()}")
print(f"Output dir:     {OUTPUT_DIR.resolve()}")
print(f"Adapter dir:    {ADAPTER_DIR.resolve()}")
print(f"Baseline file:  {BASELINE_FILE.resolve()}")

In [ ]:
# Load training_data.jsonl and format into Llama 3.1 ChatML-style text
records = []
with open(DATA_PATH, "r") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Total records loaded: {len(records)}")
print(f"Keys in each record:  {list(records[0].keys())}")

# ── Format into Llama 3.1 chat template ──
def format_chat(example):
    text = "<|begin_of_text|>"
    for msg in example["messages"]:
        role = msg["role"]
        content = msg["content"]
        if role == "user":
            text += f"<|start_header_id|>user<|end_header_id|>\n\n{content}<|eot_id|>\n\n"
        elif role == "assistant":
            text += f"<|start_header_id|>assistant<|end_header_id|>\n\n{content}<|eot_id|>\n\n"
    return {"text": text}

dataset = Dataset.from_list(records)
dataset = dataset.map(format_chat)

# ── Train / eval split (13 train, 2 eval) ──
split_dataset = dataset.train_test_split(test_size=2, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"\nTrain examples: {len(train_dataset)}")
print(f"Eval examples:  {len(eval_dataset)}")
print(f"\n--- Formatted Sample (first 400 chars) ---")
print(train_dataset[0]["text"][:400])

## Load Tokenizer and 4-Bit Quantized Model

In [ ]:
# Load tokenizer and 4-bit quantized model with QLoRA configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Tokenizer loaded.")

print(f"Loading model in 4-bit: {MODEL_ID}")
start_time = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    use_cache=False,
)
load_time = time.time() - start_time
print(f"Model loaded in {load_time:.1f}s")
print(f"Model device: {model.device}")

# Prepare for k-bit training (gradient checkpointing, input requires grad)
model = prepare_model_for_kbit_training(model)
print("Model prepared for k-bit training.")

## Compute Baseline Metrics (Before Fine-Tuning)

In [ ]:
# Compute baseline metrics BEFORE fine-tuning
# Uses ignore_index=-100 to mask padding tokens — critical for accurate loss.

def tokenize_eval(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=1024,
        return_tensors=None,
    )

eval_dataset_tokenized = eval_dataset.map(tokenize_eval, remove_columns=["text"])
eval_dataset_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])

eval_loader = DataLoader(eval_dataset_tokenized, batch_size=2, shuffle=False)

model.eval()
total_loss = 0.0
total_tokens = 0

with torch.no_grad():
    for batch in eval_loader:
        batch = {k: v.to("cuda") for k, v in batch.items()}
        batch["labels"] = batch["input_ids"].clone()
        # Mask padding positions so they are excluded from loss computation
        batch["labels"][batch["attention_mask"] == 0] = -100

        outputs = model(**batch)

        shift_logits = outputs.logits[..., :-1, :].contiguous()
        shift_labels = batch["labels"][..., 1:].contiguous()

        # ignore_index=-100 excludes padding from the sum
        loss_fct = CrossEntropyLoss(reduction="sum", ignore_index=-100)
        batch_loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        )

        total_loss += batch_loss.item()
        total_tokens += (shift_labels != -100).sum().item()

baseline_avg_loss = total_loss / total_tokens
baseline_perplexity = torch.exp(torch.tensor(baseline_avg_loss)).item()

baseline_metrics = {
    "avg_loss": round(baseline_avg_loss, 4),
    "perplexity": round(baseline_perplexity, 2),
    "total_tokens": total_tokens,
}

print("=" * 60)
print("BASELINE METRICS (Before Fine-Tuning)")
print("=" * 60)
print(f"Total non-padding tokens:  {total_tokens}")
print(f"Average loss:              {baseline_avg_loss:.4f}")
print(f"Perplexity:                {baseline_perplexity:.2f}")
print("=" * 60)

with open(BASELINE_FILE, "w") as f:
    json.dump(baseline_metrics, f, indent=2)
print(f"Baseline saved to {BASELINE_FILE}")

## Apply LoRA Configuration

In [ ]:
# Apply LoRA adapters to the quantized base model
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Log parameter breakdown
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"\nTotal parameters:       {total:,}")
print(f"Trainable params:       {trainable:,} ({100 * trainable / total:.4f}%)")
print(f"LoRA rank:              16")
print(f"LoRA alpha:             32")

## Run QLoRA Fine-Tuning

In [ ]:
# Run QLoRA training with SFTTrainer
sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    logging_steps=2,
    eval_strategy="steps",
    eval_steps=5,
    save_strategy="steps",
    save_steps=10,
    learning_rate=2e-4,
    warmup_steps=3,
    lr_scheduler_type="cosine",
    bf16=True,
    tf32=True,
    max_grad_norm=0.3,
    report_to="none",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    dataloader_num_workers=2,
    dataset_text_field="text",
    packing=False,
    max_seq_length=1024,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

print("Starting QLoRA training...")
train_result = trainer.train()
print()
print("Training completed.")

## Save LoRA Adapters

In [ ]:
# Save LoRA adapters to disk
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

adapter_size_mb = sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*")) / 1e6
print(f"LoRA adapters saved to: {ADAPTER_DIR.resolve()}")
print(f"Adapter size:            {adapter_size_mb:.1f} MB")
print(f"Files:                   {[f.name for f in ADAPTER_DIR.iterdir()]}")

## Extract Training Metrics from Checkpoint

In [ ]:
# Extract training metrics from the latest checkpoint directory
checkpoints = sorted(
    [d for d in OUTPUT_DIR.iterdir() if d.name.startswith("checkpoint-")],
    key=lambda d: int(d.name.split("-")[1]),
)

if not checkpoints:
    print("No checkpoint directories found.")
    print("Has trainer.train() been called?")
else:
    latest = checkpoints[-1]
    state_path = latest / "trainer_state.json"
    print(f"Latest checkpoint: {latest.name}")

    with open(state_path) as f:
        state = json.load(f)

    log_history = state.get("log_history", [])
    global_step = state.get("global_step", 0)

    final_loss = None
    eval_losses = []
    for entry in log_history:
        if "loss" in entry:
            final_loss = entry["loss"]
        if "eval_loss" in entry:
            eval_losses.append(entry["eval_loss"])

    print(f"Total training steps:      {global_step}")
    print(f"Epochs completed:          {state.get('epoch', 0.0):.2f}")
    if final_loss is not None:
        print(f"Final training loss:       {final_loss:.4f}")
    if eval_losses:
        print(f"Best eval loss:            {min(eval_losses):.4f}")
        print(f"Final eval loss:           {eval_losses[-1]:.4f}")
    print("=" * 60)

    print("\n--- Full Training Log ---")
    for entry in log_history:
        step = entry.get("step", "?")
        loss = entry.get("loss", "-")
        eval_loss = entry.get("eval_loss", "-")
        epoch_n = entry.get("epoch", "-")
        print(f"  Step {str(step):>3} | loss: {str(loss):>8} | eval_loss: {str(eval_loss):>8} | epoch: {str(epoch_n):>5}")

## Compare Baseline vs Fine-Tuned

In [ ]:
# Compare baseline vs fine-tuned evaluation metrics
# Reloads a clean model with 4-bit quantization and applies LoRA adapters.

torch.cuda.empty_cache()
print(f"GPU memory before reload: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

# Reload base model WITH 4-bit quantization to fit in GPU memory
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

model_ft = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model_ft.eval()
model_ft.config.use_cache = True

# Tokenize eval dataset (shorter max_length to save memory)
def tokenize_eval(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors=None,
    )

eval_dataset_tokenized = eval_dataset.map(tokenize_eval, remove_columns=["text"])
eval_dataset_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])

eval_loader = DataLoader(eval_dataset_tokenized, batch_size=1, shuffle=False)

total_loss = 0.0
total_tokens = 0

with torch.no_grad():
    for batch in eval_loader:
        batch = {k: v.to("cuda") for k, v in batch.items()}
        batch["labels"] = batch["input_ids"].clone()
        batch["labels"][batch["attention_mask"] == 0] = -100

        outputs = model_ft(**batch)

        shift_logits = outputs.logits[..., :-1, :].contiguous()
        shift_labels = batch["labels"][..., 1:].contiguous()

        loss_fct = CrossEntropyLoss(reduction="sum", ignore_index=-100)
        batch_loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        )

        total_loss += batch_loss.item()
        total_tokens += (shift_labels != -100).sum().item()

        del batch, outputs
        torch.cuda.empty_cache()

ft_avg_loss = total_loss / total_tokens
ft_perplexity = torch.exp(torch.tensor(ft_avg_loss)).item()

# Load baseline
with open(BASELINE_FILE) as f:
    baseline = json.load(f)

# Comparison output
loss_improvement = baseline["avg_loss"] - ft_avg_loss

print("\n" + "=" * 68)
print("BASELINE vs FINE-TUNED — EVALUATION COMPARISON")
print("=" * 68)
print(f"{'Metric':<20} {'Baseline':<14} {'Fine-Tuned':<14} {'Change':<14}")
print("-" * 68)
print(f"{'Avg Loss':<20} {baseline['avg_loss']:<14.4f} {ft_avg_loss:<14.4f} {loss_improvement:<+14.4f}")
print(f"{'Perplexity':<20} {baseline['perplexity']:<14.2f} {ft_perplexity:<14.2f} {baseline['perplexity'] - ft_perplexity:<+14.2f}")
print("-" * 68)
print(f"\nTotal non-padding tokens evaluated: {total_tokens}")

if loss_improvement > 0:
    print(f"\n✅ MODEL IMPROVED — Loss reduced by {loss_improvement:.4f} points ({loss_improvement / baseline['avg_loss'] * 100:.1f}% reduction)")
else:
    print(f"\n⚠️ No improvement detected. Loss increased by {abs(loss_improvement):.4f}.")

# Cleanup
del model_ft, base_model
torch.cuda.empty_cache()
print(f"\nFinal GPU allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 8: Save the LoRA Adapters and Upload to S3

In [ ]:
# Cell: Save LoRA adapters locally and upload to S3
# Estimated time: < 1 minute
# Prerequisites: S3 bucket exists, IAM role has s3:PutObject permission

import tarfile
import boto3
from pathlib import Path
from datetime import datetime

# ── Configuration ──
# Bucket created specifically for this MLA-C01 lab
BUCKET = "mla-c01-lora-520622116608"
PREFIX = "llama-qlora-finetuning"
ADAPTER_DIR = Path("./llama-3-qlora-final")

# ── Step 1: Verify the adapter directory exists ──
if not ADAPTER_DIR.exists():
    raise FileNotFoundError(
        f"Adapter directory not found at {ADAPTER_DIR.resolve()}. "
        "Run the training and save cells first."
    )

# ── Step 2: Create a timestamped tarball ──
timestamp = datetime.utcnow().strftime("%Y%m%d-%H%M%S")
adapter_tar_path = Path(f"qlora-adapter-{timestamp}.tar.gz")

with tarfile.open(str(adapter_tar_path), "w:gz") as tar:
    tar.add(str(ADAPTER_DIR), arcname="adapter")

tar_size_mb = adapter_tar_path.stat().st_size / 1e6
print(f"Tarball created:         {adapter_tar_path.name} ({tar_size_mb:.1f} MB)")

# ── Step 3: Upload to S3 using boto3 client interface ──
s3_client = boto3.client("s3", region_name="us-east-1")
s3_key = f"{PREFIX}/models/{adapter_tar_path.name}"

try:
    s3_client.upload_file(str(adapter_tar_path), BUCKET, s3_key)
    print(f"Uploaded to:             s3://{BUCKET}/{s3_key}")

    # ── Step 4: Verify the upload ──
    head_response = s3_client.head_object(Bucket=BUCKET, Key=s3_key)
    print(f"Verified:                {head_response['ContentLength'] / 1e6:.1f} MB")
    print(f"ETag:                    {head_response['ETag']}")

except Exception as e:
    print(f"Upload failed:           {e}")
    print("\nTroubleshooting:")
    print("  1. Verify the bucket exists:")
    print(f"     aws s3 ls s3://{BUCKET}")
    print("  2. Verify your IAM role has s3:PutObject permission")
    print("  3. If using a different region, update region_name above")

# ── Step 5: Clean up local tarball (optional) ──
# Uncomment the next line to remove the tarball after upload:
# adapter_tar_path.unlink()

## Step 9: Test Inference with the Fine-Tuned Model

In [ ]:
# Cell: Run inference with the fine-tuned model

import torch
from transformers import AutoTokenizer

# ── Reload tokenizer with Llama 3.1 special tokens ──
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    additional_special_tokens=[
        "<|begin_of_text|>", "<|end_of_text|>",
        "<|start_header_id|>", "<|end_header_id|>",
        "<|eot_id|>", "<|eom_id|>",
    ],
)

# ── CRITICAL: Set the Llama 3.1 Instruct chat template ──
# The base model tokenizer does not include one.
# This is the official template from meta-llama/Meta-Llama-3.1-8B-Instruct.
tokenizer.chat_template = (
    "{% set loop_messages = messages %}"
    "{% for message in loop_messages %}"
    "{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n'"
    "+ message['content'] | trim + '<|eot_id|>' %}"
    "{% if loop.index0 == 0 %}"
    "{% set content = '<|begin_of_text|>' + content %}"
    "{% endif %}"
    "{{ content }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}"
    "{% endif %}"
)

# ── Pad/eos token config ──
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id

# ── Inference ──
model.eval()

messages = [
    {"role": "user", "content": "Classify the sentiment: 'This product is amazing!'"},
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(f"Formatted input:\n{input_text!r}\n")

inputs = tokenizer(input_text, return_tensors="pt").to("cuda:0")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=64,
        temperature=0.1,
        top_p=0.9,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print(f"Generated response: {response}")

## Summary

In this lab you:

| Step | Achievement |
|------|-------------|
| **Step 1–2** | Verified the GPU environment and inspected the JSONL dataset |
| **Step 3** | Configured 4-bit NF4 quantization (BitsAndBytes) |
| **Step 4** | Applied LoRA with rank 16 across attention and MLP layers |
| **Step 5** | Formatted data using the Llama chat template |
| **Step 6** | Ran QLoRA training directly on the notebook GPU via SFTTrainer |
| **Step 7–8** | Evaluated results against baseline and saved adapters to S3 |
| **Step 9** | Validated the fine-tuned model with a test inference |

**Infrastructure efficiency:** Zero overhead — one instance, one role, no cold starts. Total VRAM usage: ~16-18 GB of 24 GB available.

**Next sprint step:** SageMaker Clarify for bias analysis (Week 2 Day 4) or RAG with Bedrock (Week 2 Day 5).